# Biohub E014 CPU association evaluation
Reuses trained E014 checkpoints and E013 image features. CPU only; no training or image inference.
Compare two decoding arms on 16 calibration videos; freeze selection before evaluating 48 development videos.
Official metric, fixed hashes, saved per-video predictions and explicit solver fallback reports. No leaderboard submission.


In [ ]:
import base64,hashlib,io,os,subprocess,sys,zipfile
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES']=''
os.environ['POLARS_PREFER_PKG']='32'
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='05d2fec1023d6376738c37eba772a260afdbd30df8c6601a9e5827e676e61e28'
package=Path('/kaggle/working/association_cpu_package');package.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        if not (package/name).resolve().is_relative_to(package.resolve()):raise ValueError(name)
    archive.extractall(package)
support=[Path('/kaggle/input/biohub-tracking-support-pack-50ep-v1'),Path('/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1')]
wheels=sorted({str(p.parent) for root in support if root.is_dir() for p in root.rglob('*.whl')})
if not wheels:raise RuntimeError('Offline dependency wheels missing')
cmd=[sys.executable,'-m','pip','install','--no-index']
for folder in wheels:cmd+=['--find-links',folder]
subprocess.run(cmd+['tracksdata','polars>=1.36','zarr>=3.0.10,<4','geff-spec<1.2'],check=True)
env=dict(os.environ,PYTHONPATH=os.pathsep.join([str(package/'src'),str(package/'scripts')]),OPENBLAS_NUM_THREADS='1',OMP_NUM_THREADS='2',MKL_NUM_THREADS='1')
subprocess.run([sys.executable,'-u',str(package/'scripts/association_cpu_runner.py'),str(package)],env=env,check=True)
